In [4]:
import polars as pl
from pathlib import Path
import os

INPUT_DIR = "../../data/processed/preprocessed_daily"

# --- Find all parquet files (one per day folder) ---
files = sorted(Path(INPUT_DIR).rglob("0.parquet"))

print(f"Found {len(files)} daily files to process.")

# --- Compute global aggregates once ---
print("Computing global aggregates (mean delay by route/stop and hour)...")
df_all = pl.scan_parquet([str(f) for f in files])

agg_route = (
    df_all.group_by(["route_id", "hod"])
    .agg(pl.col("delay").mean().alias("mean_delay_by_route_hod"))
)

agg_stop = (
    df_all.group_by(["stop_id", "hod"])
    .agg(pl.col("delay").mean().alias("mean_delay_by_stop_hod"))
)

# Compute global max of shape_dist_traveled per route_id
global_route_max = (
    df_all
    .group_by("route_id")
    .agg(pl.col("shape_dist_traveled").cast(pl.Float64).max().alias("route_max_dist"))
)

agg_route_file = "../../data/interim/utils/agg_route.parquet"
agg_stop_file = "../../data/interim/utils/agg_stop.parquet"
global_route_max_file = "../../data/interim/utils/global_route_max.parquet"
agg_route.sink_parquet(agg_route_file)
agg_stop.sink_parquet(agg_stop_file)
global_route_max.sink_parquet(global_route_max_file)
print("✅ Global aggregates saved.\n")

Found 360 daily files to process.
Computing global aggregates (mean delay by route/stop and hour)...
✅ Global aggregates saved.



In [11]:
global_route_max_df = pl.read_parquet(global_route_max_file)
print(global_route_max_df.schema)

Schema([('route_id', String), ('route_max_dist', Float64)])


In [16]:
def scan_parquet_str_date(path: str) -> pl.LazyFrame:
    return (
        pl.scan_parquet(path)
        .with_columns(
            pl.col("date")
            .cast(pl.Utf8)   # ensures it's string immediately after scanning
            .alias("date")
        )
    )


In [19]:
# Load global lookup (eager, small)
route_max_df = pl.read_parquet(global_route_max_file)
route_max_df = route_max_df.with_columns([
    pl.col("route_max_dist").cast(pl.Float64)
])

OUT_DIR = "../../data/feature_engineered/features_daily"
os.makedirs(OUT_DIR, exist_ok=True)

# for date_dir in dates:
for file in files:
    date_str = file.parent.name.split("date=")[-1]

    # df = pl.scan_parquet(str(file))
    df = scan_parquet_str_date(str(file))

    # Join with global route max
    df = df.join(route_max_df.lazy(), on="route_id", how="left")

    df = df.with_columns([
        pl.col("date").str.strptime(pl.Date, "%Y-%m-%d").alias("date_parsed"),
    ])

    # Derive features
    df = df.with_columns([
        # Normalized distance
        (pl.col("shape_dist_traveled").cast(pl.Float64) / pl.col("route_max_dist"))
            .alias("dist_from_start"),

        # Temporal features
        pl.col("date_parsed").dt.month().alias("month"),
        pl.col("date_parsed").dt.weekday().alias("day_of_week"),
        (pl.col("hod").is_between(7, 9) | pl.col("hod").is_between(15, 18)).alias("is_peak_hour"),

        # Weather interactions
        (pl.col("snow_depth") > 0).alias("is_snowy"),
        (pl.col("precipitation") > 0).alias("is_rainy"),
        (pl.col("temperature") < 0).alias("temp_below_zero"),
        (pl.col("wind_speed") > 10).alias("is_high_wind"),
        (pl.col("precipitation") * pl.col("wind_speed")).alias("precip_wind_interaction"),

        # Cyclical hour encoding
        (pl.col("hod") * (2 * 3.14159 / 24)).sin().alias("sin_hod"),
        (pl.col("hod") * (2 * 3.14159 / 24)).cos().alias("cos_hod"),
    ])

    # Join global aggregates (lazy)
    df = (
        df.join(pl.scan_parquet(agg_route_file), on=["route_id", "hod"], how="left")
          .join(pl.scan_parquet(agg_stop_file), on=["stop_id", "hod"], how="left")
    )

    df = df.with_columns([
        pl.col("delay").cast(pl.Float32),
        pl.col("route_type").cast(pl.Int32),
        pl.col("direction_id").cast(pl.Int8),
        pl.col("shape_dist_traveled").cast(pl.Float32),
        pl.col("temperature").cast(pl.Float32),
        pl.col("precipitation").cast(pl.Float32),
        pl.col("snow_depth").cast(pl.Float32),
        pl.col("wind_speed").cast(pl.Float32),
        pl.col("route_max_dist").cast(pl.Float32),
        pl.col("dist_from_start").cast(pl.Float32),
        pl.col("precip_wind_interaction").cast(pl.Float32),
        pl.col("sin_hod").cast(pl.Float32),
        pl.col("cos_hod").cast(pl.Float32),
        pl.col("mean_delay_by_route_hod").cast(pl.Float32),
        pl.col("mean_delay_by_stop_hod").cast(pl.Float32),
    ])

    # Keep only final columns
    final_cols = [
        "date", "hod", "month", "day_of_week", "is_peak_hour",
        "stop_id", "route_id", "route_type", "direction_id",
        "dist_from_start",
        "temperature", "precipitation", "snow_depth", "wind_speed",
        "is_snowy", "is_rainy", "temp_below_zero", "is_high_wind", "precip_wind_interaction",
        "sin_hod", "cos_hod", "mean_delay_by_route_hod", "mean_delay_by_stop_hod",
        "delay"
    ]

    df = df.select(final_cols)
    
    # Write per-day Parquet
    out_file = Path(OUT_DIR) / f"date={date_str}.parquet"
    df.sink_parquet(str(out_file), compression="zstd")